# Reusable Template: Multiple Linear Regression for Economics Projects

Drop in your own dataset and this notebook runs the full pipeline: load → explore → scale →
train (gradient descent, with a closed-form cross-check) → interpret coefficients
economically → diagnose fit → predict on new observations.

**How to reuse this for a new project:** edit only the **CONFIG** cell (Section 1) and the
**Load your data** cell (Section 2). Everything after that is written generically against
`X` (features matrix), `y` (target vector), and `FEATURE_NAMES`/`TARGET_NAME`, so it doesn't
need to change.

# Contents
- [ 1 - Config ](#1)
- [ 2 - Load Your Data ](#2)
- [ 3 - Explore ](#3)
- [ 4 - Reusable Regression Library ](#4)
- [ 5 - Scale Features ](#5)
- [ 6 - Train (Gradient Descent) ](#6)
- [ 7 - Cross-Check (Closed-Form OLS) ](#7)
- [ 8 - Economic Interpretation ](#8)
- [ 9 - Diagnostics ](#9)
- [ 10 - Predict on New Observations ](#10)
- [ 11 - Notes for Adapting to a New Dataset ](#11)


<a name="1"></a>
## 1 - Config

**This is the only cell you *must* edit for a brand-new project.** Everything downstream reads
from these variables.


In [ ]:
# ============================== CONFIG ==============================
# Human-readable names, used only for labels/printouts -- purely cosmetic
FEATURE_NAMES = ["population (10,000s)", "median household income ($1,000s)"]
TARGET_NAME   = "average monthly profit ($10,000s)"

# Hyperparameters for gradient descent
ALPHA      = 0.1     # learning rate -- see the cheatsheet's troubleshooting table if this diverges
NUM_ITERS  = 1000     # number of gradient descent iterations

# Random seed, used only if you keep the synthetic-data generator in Section 2
SEED = 1
# ======================================================================


<a name="2"></a>
## 2 - Load Your Data

**Replace the body of this cell** with your own data source. The only requirement:
end up with `X` (shape `(m, n)`) and `y` (shape `(m,)`) as NumPy arrays.

Common replacements:
```python
# From CSV:
import pandas as pd
df = pd.read_csv("your_file.csv")
X = df[["feature_1", "feature_2"]].to_numpy()
y = df["target_column"].to_numpy()

# From a NumPy .npz / .npy file:
data = np.load("your_data.npz")
X, y = data["X"], data["y"]
```

For demonstration (so this template also runs standalone with no files), we keep the same
synthetic economic dataset from the lab: population + income → restaurant profit.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
np.set_printoptions(precision=4, suppress=True)

def generate_economic_data(m=100, seed=SEED):
    """Placeholder data source -- swap this cell for pd.read_csv(...) on your own project."""
    rng = np.random.default_rng(seed)
    population = rng.uniform(2, 25, m)
    income     = rng.uniform(20, 120, m)
    true_w = np.array([1.15, 0.045])
    true_b = -5.2
    noise = rng.normal(0, 3.0, m)
    y = true_w[0]*population + true_w[1]*income + true_b + noise
    X = np.column_stack([population, income])
    return X, y

X, y = generate_economic_data()
m, n = X.shape
assert len(FEATURE_NAMES) == n, "FEATURE_NAMES must have one entry per column of X"
print(f"Loaded {m} examples with {n} features: {FEATURE_NAMES}")
print(f"Target: {TARGET_NAME}")


<a name="3"></a>
## 3 - Explore

Generic exploratory checks: shape, summary stats, per-feature scatter against the target, and
a feature correlation matrix (multicollinearity check). Works for any `n`.


In [ ]:
print("X shape:", X.shape, " y shape:", y.shape)
print("\nPer-feature summary (mean, std, min, max):")
for j, name in enumerate(FEATURE_NAMES):
    col = X[:, j]
    print(f"  {name:35s} mean={col.mean():8.2f}  std={col.std():8.2f}  min={col.min():8.2f}  max={col.max():8.2f}")
print(f"\n{TARGET_NAME:35s} mean={y.mean():8.2f}  std={y.std():8.2f}  min={y.min():8.2f}  max={y.max():8.2f}")

fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
if n == 1:
    axes = [axes]
for j in range(n):
    axes[j].scatter(X[:, j], y, alpha=0.6)
    axes[j].set_xlabel(FEATURE_NAMES[j])
    axes[j].set_ylabel(TARGET_NAME)
    axes[j].set_title(f"{TARGET_NAME}\nvs. {FEATURE_NAMES[j]}")
plt.tight_layout()
plt.show()

if n > 1:
    print("Feature correlation matrix (watch for multicollinearity, e.g. |corr| > 0.8):")
    print(np.corrcoef(X.T).round(3))


<a name="4"></a>
## 4 - Reusable Regression Library

Generic, dimension-agnostic functions (work for any number of features `n`). These are the
same core functions from the lab and cheatsheet, packaged for reuse without modification.


In [ ]:
def compute_cost(X, y, w, b):
    """Mean-squared-error cost (halved), vectorized for any n features."""
    m = X.shape[0]
    errors = X @ w + b - y
    return (1 / (2 * m)) * np.sum(errors ** 2)

def compute_gradient(X, y, w, b):
    """Gradients of the cost w.r.t. w (vector) and b (scalar), vectorized for any n."""
    m = X.shape[0]
    errors = X @ w + b - y
    dj_dw = (1 / m) * (X.T @ errors)
    dj_db = (1 / m) * np.sum(errors)
    return dj_dw, dj_db

def zscore_normalize_features(X):
    """Per-column z-score normalization. Returns X_norm, mu, sigma."""
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    return (X - mu) / sigma, mu, sigma

def unscale_coefficients(w_norm, b_norm, mu, sigma):
    """Converts weights/bias trained on normalized features back to raw-feature units."""
    w_raw = w_norm / sigma
    b_raw = b_norm - np.sum((w_norm * mu) / sigma)
    return w_raw, b_raw

def gradient_descent(X, y, w_in, b_in, alpha, num_iters, print_every=None):
    """Batch gradient descent. Set print_every=N to log progress every N iterations."""
    w, b = copy.deepcopy(w_in), b_in
    J_history = []
    for i in range(num_iters):
        dj_dw, dj_db = compute_gradient(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        J_history.append(compute_cost(X, y, w, b))
        if print_every and i % print_every == 0:
            print(f"Iteration {i:5d}: cost {J_history[-1]:10.4f}")
    return w, b, J_history

def ols_closed_form(X, y):
    """Exact OLS solution via the normal equation (small/medium n only)."""
    m = X.shape[0]
    X_design = np.column_stack([np.ones(m), X])
    theta = np.linalg.lstsq(X_design, y, rcond=None)[0]
    return theta[0], theta[1:]   # b, w

def r_squared(y, y_pred):
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return 1 - ss_res / ss_tot

def adjusted_r_squared(y, y_pred, n_features):
    m = len(y)
    r2 = r_squared(y, y_pred)
    return 1 - (1 - r2) * (m - 1) / (m - n_features - 1)

def elasticities(w_raw, X, y):
    """% change in y per 1% change in each feature, evaluated at the means."""
    x_bar = np.mean(X, axis=0)
    y_bar = np.mean(y)
    return w_raw * (x_bar / y_bar)

print("Regression library loaded (compute_cost, compute_gradient, zscore_normalize_features,",
      "unscale_coefficients, gradient_descent, ols_closed_form, r_squared,",
      "adjusted_r_squared, elasticities).")


<a name="5"></a>
## 5 - Scale Features

Z-score normalization before training. If you swap in a dataset with a very skewed feature
(e.g., GDP, population of very different city sizes), consider log-transforming it in Section 2
*before* this step (see the cheatsheet's Section 4 for when to prefer a log transform).


In [ ]:
X_norm, mu, sigma = zscore_normalize_features(X)
print("Feature means:", dict(zip(FEATURE_NAMES, mu.round(3))))
print("Feature std devs:", dict(zip(FEATURE_NAMES, sigma.round(3))))


<a name="6"></a>
## 6 - Train (Gradient Descent)


In [ ]:
w_init = np.zeros(n)
b_init = 0.

w_final, b_final, J_history = gradient_descent(
    X_norm, y, w_init, b_init, ALPHA, NUM_ITERS, print_every=max(1, NUM_ITERS // 10)
)

plt.plot(J_history)
plt.xlabel("iteration"); plt.ylabel("cost J(w,b)")
plt.title("Training curve"); plt.show()

print("\nFinal cost:", J_history[-1])


<a name="7"></a>
## 7 - Cross-Check (Closed-Form OLS)

Only meaningful for **plain linear regression** (no regularization/nonlinearity). If you extend
this template with regularization or polynomial features, this cross-check no longer applies
directly — see the cheatsheet's Section 6 for why.


In [ ]:
b_closed, w_closed = ols_closed_form(X, y)
w_gd_raw, b_gd_raw = unscale_coefficients(w_final, b_final, mu, sigma)

print("Gradient descent (raw units):", dict(zip(FEATURE_NAMES, w_gd_raw.round(4))), " b =", round(b_gd_raw, 4))
print("Closed-form OLS  (raw units):", dict(zip(FEATURE_NAMES, w_closed.round(4))), " b =", round(b_closed, 4))
print("\nThese should agree closely -- if not, revisit ALPHA/NUM_ITERS in the config cell.")


<a name="8"></a>
## 8 - Economic Interpretation

Prints marginal effects (raw units), standardized coefficients, and elasticities — the three
standard ways an economist would report and compare regression coefficients.


In [ ]:
elast = elasticities(w_gd_raw, X, y)

print(f"{'Feature':35s}{'Marginal effect (raw)':>24s}{'Standardized coef':>20s}{'Elasticity':>14s}")
for j, name in enumerate(FEATURE_NAMES):
    print(f"{name:35s}{w_gd_raw[j]:>24.4f}{w_final[j]:>20.4f}{elast[j]:>14.3f}")
print(f"\nIntercept (raw units): {b_gd_raw:.4f}")
print(f"\nReading a marginal effect: a 1-unit increase in a feature is associated with a change\n"
      f"of that many units in '{TARGET_NAME}', holding the other feature(s) fixed.")
print(f"Reading an elasticity: a 1% increase in a feature is associated with roughly that many\n"
      f"percent change in '{TARGET_NAME}', evaluated at the sample means.")


<a name="9"></a>
## 9 - Diagnostics


In [ ]:
y_pred = X_norm @ w_final + b_final
r2 = r_squared(y, y_pred)
adj_r2 = adjusted_r_squared(y, y_pred, n) if m > n + 1 else float('nan')
print(f"R-squared: {r2:.4f}")
print(f"Adjusted R-squared: {adj_r2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
lims = [min(y.min(), y_pred.min()), max(y.max(), y_pred.max())]
axes[0].scatter(y, y_pred, alpha=0.6)
axes[0].plot(lims, lims, 'r--')
axes[0].set_xlabel(f"Actual {TARGET_NAME}")
axes[0].set_ylabel(f"Predicted {TARGET_NAME}")
axes[0].set_title(f"Actual vs. Predicted (R²={r2:.3f})")

residuals = y - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.6)
axes[1].axhline(0, color='r', linestyle='--')
axes[1].set_xlabel(f"Predicted {TARGET_NAME}")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residual plot (should look like noise around 0)")

plt.tight_layout()
plt.show()


<a name="10"></a>
## 10 - Predict on New Observations

Edit `new_observations` with your own feature values, in the **same order as `FEATURE_NAMES`**.


In [ ]:
def predict(x_raw, mu, sigma, w, b):
    x_raw = np.asarray(x_raw, dtype=float)
    x_norm = (x_raw - mu) / sigma
    return x_norm @ w + b

# Edit these rows for your own project -- each row must have n values, matching FEATURE_NAMES order
new_observations = [
    [15.0, 60.0],
    [5.0, 100.0],
    [22.0, 30.0],
]

for obs in new_observations:
    pred = predict(obs, mu, sigma, w_final, b_final)
    obs_str = ", ".join(f"{name}={val}" for name, val in zip(FEATURE_NAMES, obs))
    print(f"{obs_str}  ->  predicted {TARGET_NAME}: {pred:.4f}")


<a name="11"></a>
## 11 - Notes for Adapting to a New Dataset

- **Different number of features?** Nothing needs to change except `FEATURE_NAMES` (Section 1)
  and how you build `X` (Section 2) — every function in Section 4 is written generically for
  any `n`.
- **Categorical variables** (e.g., "region")? One-hot encode them into 0/1 columns before
  building `X`; don't feed raw category labels into the regression.
- **Very skewed features** (income, population, GDP, firm revenue)? Consider `np.log1p(x)`
  before scaling — this is standard practice in applied econometrics and pairs naturally with
  the elasticity interpretation in Section 8.
- **Suspect nonlinearity or diminishing returns?** Add engineered columns (e.g., `x**2`,
  `x1*x2`) to `X` before Section 5 — the pipeline doesn't care whether a column is an original
  feature or an engineered one.
- **Want regularization (Ridge)?** Add `+ (lam/(2*m))*np.sum(w**2)` inside `compute_cost` and
  `+ (lam/m)*w` inside `compute_gradient`'s `dj_dw` — see Extension Question 3 in the lab
  notebook for the full derivation.
- **Multiple candidate models?** Loop Sections 5-9 over different feature subsets and compare
  `adjusted_r_squared` (not plain `r_squared`, which never decreases as you add features).
